# Gold — Corridor Stats

Aggregates silver journeys into performance analytics:
- Overall corridor stats (avg / best / worst)
- Avg corridor time by hour of day
- Avg corridor time by day of week
- Segment variability — which stretch causes the most delay?

Requires `datastreaming/data/silver_journeys.parquet` (run silver notebook first).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SILVER = Path("../data/silver_journeys.parquet")
df = pd.read_parquet(SILVER)
print(f"Loaded {len(df):,} journeys")

pd.set_option("display.float_format", "{:.1f}".format)

## Overall corridor stats

In [ ]:
stats = df["total_minutes"].agg(["mean", "min", "max", "std", "count"])
print(f"Avg:    {stats['mean']:.1f} min")
print(f"Best:   {stats['min']:.1f} min")
print(f"Worst:  {stats['max']:.1f} min")
print(f"Std:    {stats['std']:.1f} min")
print(f"Total:  {int(stats['count']):,} journeys")

## By hour of day

In [ ]:
by_hour = df.groupby("hour")["total_minutes"].agg(["mean", "min", "max", "count"])
by_hour.columns = ["avg", "best", "worst", "count"]
print(by_hour.round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_hour.index, by_hour["avg"], color="steelblue", label="avg")
ax.plot(by_hour.index, by_hour["worst"], "r--", label="worst", linewidth=1)
ax.plot(by_hour.index, by_hour["best"],  "g--", label="best",  linewidth=1)
ax.set_xlabel("Hour of day")
ax.set_ylabel("Minutes")
ax.set_title("Gh. Sincai → Piața Română — corridor time by hour")
ax.legend()
plt.tight_layout()
plt.show()

## By day of week

In [ ]:
ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df.groupby("day_of_week")["total_minutes"].agg(["mean", "count"]).reindex(ORDER)
by_day.columns = ["avg_min", "count"]
print(by_day.round(1))

by_day["avg_min"].plot(kind="bar", figsize=(8, 4), color="steelblue",
                       title="Avg corridor time by day of week", ylabel="minutes")
plt.tight_layout()
plt.show()

## Segment variability — where do delays happen?

In [ ]:
SEG_COLS = [
    "sincai_to_marasesti_min",
    "marasesti_to_sf_gheorghe_min",
    "sf_gheorghe_to_universitate_min",
    "universitate_to_balcescu_min",
    "balcescu_to_verona_min",
    "verona_to_romana_min",
]
seg = df[SEG_COLS].agg(["mean", "std"]).T.round(2)
seg.columns = ["avg_min", "std_min"]
seg.index = [c.replace("_min", "").replace("_to_", " → ") for c in seg.index]
seg = seg.sort_values("std_min", ascending=False)
print("Segments ranked by variability (std):")
print(seg)

seg.plot(kind="bar", figsize=(10, 4),
         title="Segment avg time and variability", ylabel="minutes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()